In [1]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt, seaborn as sns
from ss_utils import *
df = load_data(".")   # the folder with train.csv and test.csv

In [2]:
import json
X, y, g, X_te, ids, _ = build(df, impute="logic", groups=("G1", "G2", "G3", "G4"))
tp = pd.read_csv("results/tabpfn_test_probs.csv")              # TabPFN + CatBoost test probabilities from Colab
assert (tp.PassengerId.to_numpy() == ids).all()
path = make_submission(ids, (tp.tabpfn + tp.cat).to_numpy() / 2, "FINAL_tabpfn_cat_blend")

sub, sample = pd.read_csv(path), pd.read_csv("sample_submission.csv")
assert list(sub.columns) == list(sample.columns)
assert len(sub) == len(sample) == 4277 and (sub.PassengerId == sample.PassengerId).all()
same = (sub.Transported == pd.read_csv("submissions/E5_tabpfn_cat_blend.csv").Transported).all()
print("identical to E5_tabpfn_cat_blend.csv:", same, "| predicted True share:", round(sub.Transported.mean(), 3))

Saved submissions/FINAL_tabpfn_cat_blend.csv: 4277 rows, 0.513 predicted True
identical to E5_tabpfn_cat_blend.csv: True | predicted True share: 0.513


Final Kaggle Leaderboard:
Rank = 79
Total number of teams = 1895
Percentile = 79/1895 * 100% = 4.17%

In [3]:
log = pd.read_csv(LOG_PATH)
log.sort_values(["experiment", "run_id"]).to_csv("results/all_runs_summary.csv", index=False)

final_cfg = pd.DataFrame([
    {"Component": "Features", "Setting": "logic imputation + groups G1–G4 (28 features; 40 after one-hot for TabPFN)"},
    {"Component": "CatBoost", "Setting": "library defaults (1,000 trees, depth 6, automatic learning rate), mean of 5 seeds"},
    {"Component": "TabPFN", "Setting": "TabPFNClassifier defaults, Colab T4 GPU, random_state=0"},
    {"Component": "Blend", "Setting": "mean of the two probabilities, threshold 0.5"}])
final_cfg.to_latex("results/final_model.tex", index=False)

best = {m: json.load(open(f"results/best_{m}.json")) for m in ["lgbm", "xgb", "cat"]}
best["cat_3seed"] = json.load(open("results/best_cat_3seed.json"))
pd.DataFrame(best).to_latex("results/tuned_hyperparameters.tex", na_rep="—")

import glob; print(sorted(glob.glob("figures/*.png"))); print(sorted(glob.glob("results/*.tex")))

['figures/fig_age.png', 'figures/fig_blend_weight.png', 'figures/fig_cabin_num.png', 'figures/fig_corr.png', 'figures/fig_cv_vs_lb.png', 'figures/fig_missing.png', 'figures/fig_oof_corr.png', 'figures/fig_optuna_lgbm.png', 'figures/fig_rate_by_cat.png', 'figures/fig_shap_summary.png', 'figures/fig_spend.png', 'figures/fig_target.png']
['results/e1_cumulative.tex', 'results/e1_leave_one_out.tex', 'results/e2_imputation.tex', 'results/e4_models.tex', 'results/e5_candidates.tex', 'results/e5_disagreement.tex', 'results/e5_ensembles.tex', 'results/e5_final_candidates.tex', 'results/e6_shap_top15.tex', 'results/e7_threshold.tex', 'results/e8_submission_overlap.tex', 'results/error_analysis.tex', 'results/final_model.tex', 'results/tuned_hyperparameters.tex']
